In [78]:
pd.read_sql("PRAGMA table_info(companies);", conn)

,cid,name,type,notnull,dflt_value,pk
0,0,id,TEXT,0,None,0
1,1,company_logo,TEXT,0,None,0
2,2,company_name,TEXT,0,None,0
3,3,chart_link,TEXT,0,None,0
4,4,about_company,TEXT,0,None,0
5,5,website,TEXT,0,None,0
6,6,nse_profile,TEXT,0,None,0
7,7,bse_profile,TEXT,0,None,0
8,8,face_value,REAL,0,None,0
9,9,book_value,REAL,0,None,0


In [77]:
import pandas as pd
import sqlite3

conn = sqlite3.connect("../db/nifty100.db")
df = pd.read_sql("SELECT * FROM financial_ratios", conn)
companies = pd.read_sql("SELECT index AS company_id, broad_sector FROM companies", conn)
df = df.merge(companies, on="company_id", how="left")
print(df[["company_id", "broad_sector"]].head())

DatabaseError: Execution failed on sql 'SELECT index AS company_id, broad_sector FROM companies': near "index": syntax error

In [2]:
df = df.rename(columns={"index": "company_id"})

In [3]:
import sys
from pathlib import Path
project_root = Path.cwd().parent

if str(project_root) not in sys.path:
    sys.path.append(str(project_root))

In [4]:

print(df.columns)

Index(['id_x', 'company_id', 'year', 'sales', 'expenses', 'operating_profit',
       'opm_percentage', 'other_income', 'interest', 'depreciation',
       'profit_before_tax', 'tax_percentage', 'net_profit', 'eps',
       'dividend_payout', 'id_y', 'equity_capital', 'reserves', 'borrowings',
       'other_liabilities', 'total_liabilities', 'fixed_assets', 'cwip',
       'investments', 'other_asset', 'total_assets'],
      dtype='str')


In [5]:
print(df.columns)

Index(['id_x', 'company_id', 'year', 'sales', 'expenses', 'operating_profit',
       'opm_percentage', 'other_income', 'interest', 'depreciation',
       'profit_before_tax', 'tax_percentage', 'net_profit', 'eps',
       'dividend_payout', 'id_y', 'equity_capital', 'reserves', 'borrowings',
       'other_liabilities', 'total_liabilities', 'fixed_assets', 'cwip',
       'investments', 'other_asset', 'total_assets'],
      dtype='str')


# LEVERAGE & EFFICIENCY RATIOS

In [6]:
def debt_to_equity(borrowings, equity, reserves):
    total_equity = equity + reserves
    if borrowings == 0:
        return 0   # important rule
    if total_equity <= 0:
        return None
    return borrowings / total_equity


def interest_coverage(op_profit, other_income, interest):
    if interest == 0:
        return None
    return (op_profit + other_income) / interest


def asset_turnover(sales, total_assets):
    if total_assets == 0:
        return None
    return sales / total_assets

In [7]:
#apply to df
df["de_ratio"] = df.apply(
    lambda x: debt_to_equity(
        x["borrowings"],
        x["equity_capital"],
        x["reserves"]
    ), axis=1
)

df["interest_coverage"] = df.apply(
    lambda x: interest_coverage(
        x["operating_profit"],
        x["other_income"],
        x["interest"]
    ), axis=1
)

df["asset_turnover"] = df.apply(
    lambda x: asset_turnover(
        x["sales"],
        x["total_assets"]
    ), axis=1
)

In [8]:
#flags for sprint
# High 
df["high_leverage_flag"] = df["de_ratio"].apply(
    lambda x: True if x is not None and x > 5 else False
)

# ICR label
df["icr_label"] = df["interest_coverage"].apply(
    lambda x: "Debt Free" if x is None else "Normal"
)

# ICR risk
df["icr_risk_flag"] = df["interest_coverage"].apply(
    lambda x: True if x is not None and x < 1.5 else False
)

In [9]:
#veriyyy
df[[
    "company_id",
    "year",
    "de_ratio",
    "interest_coverage",
    "asset_turnover",
    "high_leverage_flag",
    "icr_label"
]].head()

,company_id,year,de_ratio,interest_coverage,asset_turnover,high_leverage_flag,icr_label
0,ABB,Dec 2012,0.0,NaN,1.822492,False,Normal
1,ABB,Mar 2014,0.0,NaN,1.998244,False,Normal
2,ABB,Mar 2015,0.0,NaN,1.665939,False,Normal
3,ABB,Mar 2016,0.0,138.333333,1.617574,False,Normal
4,ABB,Mar 2017,0.0,227.500000,1.405131,False,Normal


# CAGR Function

In [10]:
def calculate_cagr(start, end, years):
    if start is None or end is None:
        return None, "MISSING"
    if years <= 0:
        return None, "INVALID_YEARS"

    # Edge cases
    if start == 0:
        return None, "ZERO_BASE"

    if start > 0 and end > 0:
        cagr = ((end / start) ** (1 / years) - 1) * 100
        return cagr, "NORMAL"

    if start > 0 and end < 0:
        return None, "DECLINE_TO_LOSS"

    if start < 0 and end > 0:
        return None, "TURNAROUND"

    if start < 0 and end < 0:
        return None, "BOTH_NEGATIVE"

    return None, "UNKNOWN"

In [11]:
# preparing dataaa
df = df.sort_values(["company_id", "year"])

In [12]:
# for 5 years revenue
def compute_revenue_cagr(group):
    group = group.sort_values("year")
    if len(group) < 5:
        return pd.Series([None, "INSUFFICIENT"])

    start = group.iloc[-5]["sales"]
    end = group.iloc[-1]["sales"]

    return pd.Series(calculate_cagr(start, end, 5))

cagr_df = df.groupby("company_id").apply(compute_revenue_cagr)
cagr_df.columns = ["revenue_cagr_5yr", "revenue_cagr_5yr_flag"]

cagr_df.head()

,revenue_cagr_5yr,revenue_cagr_5yr_flag
company_id,,
ABB,7.400914,NORMAL
ADANIENSOL,7.784285,NORMAL
ADANIENT,17.308742,NORMAL
ADANIGREEN,29.321942,NORMAL
ADANIPORTS,9.305585,NORMAL


In [71]:
#merging baack
df = df.merge(cagr_df, on="company_id", how="left")

In [14]:
#checking
df[["company_id", "year", "sales", "revenue_cagr_5yr", "revenue_cagr_5yr_flag"]].head(7)

,company_id,year,sales,revenue_cagr_5yr,revenue_cagr_5yr_flag
0,ABB,Dec 2012,1653,7.400914,NORMAL
1,ABB,Mar 2014,2276,7.400914,NORMAL
2,ABB,Mar 2015,2289,7.400914,NORMAL
3,ABB,Mar 2016,2614,7.400914,NORMAL
4,ABB,Mar 2017,2903,7.400914,NORMAL
5,ABB,Mar 2018,3298,7.400914,NORMAL
6,ABB,Mar 2019,3679,7.400914,NORMAL


In [15]:
from src.analytics.cagr import calculate_cagr

# pat cagr

In [16]:
def pat_cagr_5yr(group):
    group = group.sort_values("year")

    cagr_list = []
    flag_list = []

    for i in range(len(group)):
        if i < 5:
            cagr_list.append(None)
            flag_list.append("INSUFFICIENT")
        else:
            start = group.iloc[i-5]["net_profit"]
            end = group.iloc[i]["net_profit"]

            cagr, flag = calculate_cagr(start, end, 5)
            cagr_list.append(cagr)
            flag_list.append(flag)

    group["pat_cagr_5yr"] = cagr_list
    group["pat_cagr_5yr_flag"] = flag_list

    return group

In [17]:
print(df.columns)

Index(['id_x', 'company_id', 'year', 'sales', 'expenses', 'operating_profit',
       'opm_percentage', 'other_income', 'interest', 'depreciation',
       'profit_before_tax', 'tax_percentage', 'net_profit', 'eps',
       'dividend_payout', 'id_y', 'equity_capital', 'reserves', 'borrowings',
       'other_liabilities', 'total_liabilities', 'fixed_assets', 'cwip',
       'investments', 'other_asset', 'total_assets', 'de_ratio',
       'interest_coverage', 'asset_turnover', 'high_leverage_flag',
       'icr_label', 'icr_risk_flag', 'revenue_cagr_5yr',
       'revenue_cagr_5yr_flag'],
      dtype='str')


In [18]:
df = df.groupby("company_id").apply(pat_cagr_5yr)

In [19]:
df[["id_x", "year", "net_profit", "pat_cagr_5yr"]].head(10)

id_x      year  net_profit pat_cagr_5yr
company_id                                           
ABB        0    61  Dec 2012         145          NaN
           1    62  Mar 2014         198          NaN
           2    63  Mar 2015         229          NaN
           3    64  Mar 2016         255          NaN
           4    65  Mar 2017         277          NaN
           5    66  Mar 2018         401     22.56184
           6    67  Mar 2019         450     17.84454
           7    68  Mar 2020         593    20.960575
           8    69  Mar 2021         691    22.063993
           9    70  Mar 2022         799    23.598557

In [20]:
df = df.rename(columns={"id_x": "company_id"})
df = df.drop(columns=["id_y"], errors="ignore")

In [21]:
df[["company_id", "year", "net_profit", "pat_cagr_5yr"]].head(10)

company_id      year  net_profit pat_cagr_5yr
company_id                                                 
ABB        0          61  Dec 2012         145          NaN
           1          62  Mar 2014         198          NaN
           2          63  Mar 2015         229          NaN
           3          64  Mar 2016         255          NaN
           4          65  Mar 2017         277          NaN
           5          66  Mar 2018         401     22.56184
           6          67  Mar 2019         450     17.84454
           7          68  Mar 2020         593    20.960575
           8          69  Mar 2021         691    22.063993
           9          70  Mar 2022         799    23.598557

# eps cagr

In [22]:
def compute_eps_cagr(group):
    group = group.sort_values("year")
    
    if len(group) < 5:
        group["eps_cagr_5yr"] = None
        group["eps_cagr_5yr_flag"] = "INSUFFICIENT"
        return group
    
    start = group.iloc[0]["eps"]
    end = group.iloc[-1]["eps"]
    
    value, flag = calculate_cagr(start, end, 5)
    
    group["eps_cagr_5yr"] = value
    group["eps_cagr_5yr_flag"] = flag
    
    return group

In [23]:
print(df.columns)
print(df.head())

Index(['company_id', 'year', 'sales', 'expenses', 'operating_profit',
       'opm_percentage', 'other_income', 'interest', 'depreciation',
       'profit_before_tax', 'tax_percentage', 'net_profit', 'eps',
       'dividend_payout', 'equity_capital', 'reserves', 'borrowings',
       'other_liabilities', 'total_liabilities', 'fixed_assets', 'cwip',
       'investments', 'other_asset', 'total_assets', 'de_ratio',
       'interest_coverage', 'asset_turnover', 'high_leverage_flag',
       'icr_label', 'icr_risk_flag', 'revenue_cagr_5yr',
       'revenue_cagr_5yr_flag', 'pat_cagr_5yr', 'pat_cagr_5yr_flag'],
      dtype='str')
              company_id      year  sales  expenses  operating_profit  \
company_id                                                              
ABB        0          61  Dec 2012   1653      1451             202.0   
           1          62  Mar 2014   2276      2009             267.0   
           2          63  Mar 2015   2289      1977             312.0   
       

In [26]:
df.rename(columns={"index": "company_id"}, inplace=True)

In [37]:
df = df.reset_index()

In [33]:
df = df.groupby("index", group_keys=False).apply(compute_eps_cagr)

In [35]:
df = df.reset_index(drop=True)

In [38]:
df[["index", "year", "eps", "eps_cagr_5yr"]].head()

,index,year,eps,eps_cagr_5yr
0,0,Dec 2012,68.0,None
1,1,Mar 2014,93.0,None
2,2,Mar 2015,108.0,None
3,3,Mar 2016,120.0,None
4,4,Mar 2017,130.0,None


In [39]:
# restore company_id if missing
if "company_id" not in df.columns:
    if "index" in df.columns:
        df.rename(columns={"index": "company_id"}, inplace=True)
    elif "id_x" in df.columns:
        df["company_id"] = df["id_x"]

print(df.columns)

Index(['index', 'company_id', 'year', 'sales', 'expenses', 'operating_profit',
       'opm_percentage', 'other_income', 'interest', 'depreciation',
       'profit_before_tax', 'tax_percentage', 'net_profit', 'eps',
       'dividend_payout', 'equity_capital', 'reserves', 'borrowings',
       'other_liabilities', 'total_liabilities', 'fixed_assets', 'cwip',
       'investments', 'other_asset', 'total_assets', 'de_ratio',
       'interest_coverage', 'asset_turnover', 'high_leverage_flag',
       'icr_label', 'icr_risk_flag', 'revenue_cagr_5yr',
       'revenue_cagr_5yr_flag', 'pat_cagr_5yr', 'pat_cagr_5yr_flag',
       'eps_cagr_5yr', 'eps_cagr_5yr_flag'],
      dtype='str')


# rank companies based on latest data


In [40]:
df_latest = df.sort_values("year").groupby("company_id").tail(1)

In [41]:
#Create SCORE
df_latest["score"] = (
    df_latest["revenue_cagr_5yr"] * 0.3 +
    df_latest["pat_cagr_5yr"] * 0.4 +
    df_latest["eps_cagr_5yr"] * 0.3
)

In [42]:
#Rank companies
df_latest["rank"] = df_latest["score"].rank(ascending=False)

In [ ]:
#Top 

top_companies = df_latest.sort_values("rank").head(10)
top_companies

,index,company_id,year,sales,expenses,operating_profit,opm_percentage,other_income,interest,depreciation,...,icr_label,icr_risk_flag,revenue_cagr_5yr,revenue_cagr_5yr_flag,pat_cagr_5yr,pat_cagr_5yr_flag,eps_cagr_5yr,eps_cagr_5yr_flag,score,rank
0,0,61,Dec 2012,1653,1451,202.0,12.0,33,0,19,...,Normal,False,7.400914,NORMAL,NaN,INSUFFICIENT,None,INSUFFICIENT,NaN,NaN
383,383,477,Dec 2012,6390,5840,550.0,9.0,136,4,82,...,Normal,False,12.554772,NORMAL,NaN,INSUFFICIENT,None,INSUFFICIENT,NaN,NaN
239,239,321,Dec 2012,8659,7310,1350.0,16.0,369,6,367,...,Normal,False,11.190310,NORMAL,NaN,INSUFFICIENT,None,INSUFFICIENT,NaN,NaN
79,79,159,Dec 2012,9749,7276,2473.0,25.0,69,78,569,...,Normal,False,4.952158,NORMAL,NaN,INSUFFICIENT,None,INSUFFICIENT,NaN,NaN
780,780,926,Dec 2013,9101,7153,1948.0,21.0,97,37,330,...,Normal,False,12.813312,NORMAL,NaN,INSUFFICIENT,None,INSUFFICIENT,NaN,NaN
384,384,478,Dec 2013,6810,6094,715.0,11.0,93,8,130,...,Normal,False,12.554772,NORMAL,NaN,INSUFFICIENT,None,INSUFFICIENT,NaN,NaN
80,80,160,Dec 2013,9161,7492,1669.0,18.0,390,67,494,...,Normal,False,4.952158,NORMAL,NaN,INSUFFICIENT,None,INSUFFICIENT,NaN,NaN
240,240,322,Dec 2013,8757,7468,1289.0,15.0,353,0,384,...,Normal,False,11.190310,NORMAL,NaN,INSUFFICIENT,None,INSUFFICIENT,NaN,NaN
81,81,161,Dec 2014,9955,8022,1934.0,19.0,419,66,513,...,Normal,False,4.952158,NORMAL,NaN,INSUFFICIENT,None,INSUFFICIENT,NaN,NaN
781,781,927,Dec 2014,9855,7816,2039.0,21.0,87,14,338,...,Normal,False,12.813312,NORMAL,NaN,INSUFFICIENT,None,INSUFFICIENT,NaN,NaN


In [44]:
#clean o/p
top_companies[[
    "company_id",
    "year",
    "revenue_cagr_5yr",
    "pat_cagr_5yr",
    "eps_cagr_5yr",
    "score",
    "rank"
]]

,company_id,year,revenue_cagr_5yr,pat_cagr_5yr,eps_cagr_5yr,score,rank
0,61,Dec 2012,7.400914,NaN,None,NaN,NaN
383,477,Dec 2012,12.554772,NaN,None,NaN,NaN
239,321,Dec 2012,11.190310,NaN,None,NaN,NaN
79,159,Dec 2012,4.952158,NaN,None,NaN,NaN
780,926,Dec 2013,12.813312,NaN,None,NaN,NaN
384,478,Dec 2013,12.554772,NaN,None,NaN,NaN
80,160,Dec 2013,4.952158,NaN,None,NaN,NaN
240,322,Dec 2013,11.190310,NaN,None,NaN,NaN
81,161,Dec 2014,4.952158,NaN,None,NaN,NaN
781,927,Dec 2014,12.813312,NaN,None,NaN,NaN


In [45]:
#load cashflow data
cf = pd.read_sql("SELECT * FROM cashflow", conn)

In [50]:
#merge df

base_keys = pl.merge(
    bs,
    on=["company_id", "year"],
    how="inner"
)[["company_id", "year"]].reset_index(drop=True)

df = df.copy()
row_positions = pd.to_numeric(df["index"], errors="raise").astype(int)

if not row_positions.between(0, len(base_keys) - 1).all():
    raise IndexError("Invalid row position found in df['index'].")

df["company_id"] = (
    base_keys.iloc[row_positions.to_numpy()]["company_id"].to_numpy()
)

def normalize_period(value):
    value = str(value).replace("-", " ")
    parts = value.split()

    if len(parts) == 2 and len(parts[1]) == 2:
        parts[1] = "20" + parts[1]

    return " ".join(parts)

df["_company_id"] = df["company_id"].astype(str).str.strip()
df["_year"] = df["year"].map(normalize_period)

cf_merge = cf.drop(columns=["id"]).copy()
cf_merge["_company_id"] = cf_merge["company_id"].astype(str).str.strip()
cf_merge["_year"] = cf_merge["year"].map(normalize_period)

df = (
    df.merge(
        cf_merge.drop(columns=["company_id", "year"]),
        on=["_company_id", "_year"],
        how="left"
    )
    .drop(columns=["_company_id", "_year"])
)

In [51]:
#cash flow
df["free_cash_flow"] = df["operating_activity"] + df["investing_activity"]

In [52]:
#cfo quantity
df["cfo_pat_ratio"] = df["operating_activity"] / df["net_profit"]

In [53]:
#avg and classification
df["cfo_quality_score"] = (
    df.groupby("company_id")["cfo_pat_ratio"]
    .transform(lambda x: x.rolling(5).mean())
)
def classify_cfo(x):
    if pd.isna(x):
        return None
    elif x > 1:
        return "HIGH_QUALITY"
    elif x >= 0.5:
        return "MODERATE"
    else:
        return "ACCRUAL_RISK"

df["cfo_quality_flag"] = df["cfo_quality_score"].apply(classify_cfo)

In [54]:
#capex intencse and classi
df["capex_intensity"] = abs(df["investing_activity"]) / df["sales"] * 100
def capex_type(x):
    if pd.isna(x):
        return None
    elif x < 3:
        return "ASSET_LIGHT"
    elif x <= 8:
        return "MODERATE"
    else:
        return "CAPITAL_INTENSIVE"

df["capex_flag"] = df["capex_intensity"].apply(capex_type)


In [55]:
#cfc converstion
df["fcf_conversion"] = df["free_cash_flow"] / df["operating_profit"] * 100

In [56]:
#
df["cfo_sign"] = df["operating_activity"].apply(lambda x: "+" if x > 0 else "-")
df["cfi_sign"] = df["investing_activity"].apply(lambda x: "+" if x > 0 else "-")
df["cff_sign"] = df["financing_activity"].apply(lambda x: "+" if x > 0 else "-")

In [57]:
#Pattern classifier:
def classify_pattern(row):
    key = (row["cfo_sign"], row["cfi_sign"], row["cff_sign"])

    if key == ("+", "-", "-"):
        return "REINVESTOR"
    elif key == ("+", "+", "-"):
        return "LIQUIDATING_ASSETS"
    elif key == ("-", "+", "+"):
        return "DISTRESS"
    elif key == ("-", "-", "+"):
        return "DEBT_FUNDED_GROWTH"
    elif key == ("+", "+", "+"):
        return "CASH_ACCUMULATOR"
    elif key == ("-", "-", "-"):
        return "PRE_REVENUE"
    elif key == ("+", "-", "+"):
        return "MIXED"
    else:
        return "OTHER"

df["capital_pattern"] = df.apply(classify_pattern, axis=1)

In [58]:
print(df.columns)

Index(['index', 'company_id', 'year', 'sales', 'expenses', 'operating_profit',
       'opm_percentage', 'other_income', 'interest', 'depreciation',
       'profit_before_tax', 'tax_percentage', 'net_profit', 'eps',
       'dividend_payout', 'equity_capital', 'reserves', 'borrowings',
       'other_liabilities', 'total_liabilities', 'fixed_assets', 'cwip',
       'investments', 'other_asset', 'total_assets', 'de_ratio',
       'interest_coverage', 'asset_turnover', 'high_leverage_flag',
       'icr_label', 'icr_risk_flag', 'revenue_cagr_5yr',
       'revenue_cagr_5yr_flag', 'pat_cagr_5yr', 'pat_cagr_5yr_flag',
       'eps_cagr_5yr', 'eps_cagr_5yr_flag', 'operating_activity',
       'investing_activity', 'financing_activity', 'net_cash_flow',
       'free_cash_flow', 'cfo_pat_ratio', 'cfo_quality_score',
       'cfo_quality_flag', 'capex_intensity', 'capex_flag', 'fcf_conversion',
       'cfo_sign', 'cfi_sign', 'cff_sign', 'capital_pattern'],
      dtype='str')


In [59]:
#save to csv output
# save to CSV output
output_dir = Path("output")
output_dir.mkdir(parents=True, exist_ok=True)

df[[
    "company_id",
    "year",
    "cfo_sign",
    "cfi_sign",
    "cff_sign",
    "capital_pattern"
]].to_csv(output_dir / "capital_allocation.csv", index=False)

In [60]:
#store to db
df = df.drop(columns=["level_0"], errors="ignore")

In [61]:
#kpi
cols = [
    "company_id",
    "year",
    "net_profit",
    "eps",
    "de_ratio",
    "interest_coverage",
    "asset_turnover",
    "revenue_cagr_5yr",
    "pat_cagr_5yr",
    "eps_cagr_5yr",
    "free_cash_flow",
    "capex_intensity",
    "cfo_quality_score"
]

df[cols].to_sql("financial_ratios", conn, if_exists="replace", index=False)

1210

In [62]:
#verify
pd.read_sql("SELECT COUNT(*) FROM financial_ratios", conn)

,COUNT(*)
0,1210


In [64]:
#13 Suppress high leverage for Financial companies
df["computed_roe"] = (df["net_profit"] / (df["equity_capital"] + df["reserves"])) * 100
df["roe_diff"] = abs(df["computed_roe"] - df.get("roe_percentage", 0))
issues = df[df["roe_diff"] > 5]
issues.to_csv("output/ratio_edge_cases.log", index=False)

In [67]:
#14 checks
df.isnull().sum()

index                       0
company_id                  0
year                        0
sales                       0
expenses                    0
operating_profit           60
opm_percentage             61
other_income                0
interest                    0
depreciation                0
profit_before_tax           0
tax_percentage              0
net_profit                  0
eps                         4
dividend_payout             4
equity_capital              0
reserves                    0
borrowings                  0
other_liabilities           0
total_liabilities           0
fixed_assets                0
cwip                        0
investments                 0
other_asset                 0
total_assets                0
de_ratio                    0
interest_coverage          93
asset_turnover              1
high_leverage_flag          0
icr_label                   0
icr_risk_flag               0
revenue_cagr_5yr            2
revenue_cagr_5yr_flag       0
pat_cagr_5

In [68]:
#businnes check
df[(df["computed_roe"] > 15) & (df["de_ratio"] < 1)].head()

,index,company_id,year,sales,expenses,operating_profit,opm_percentage,other_income,interest,depreciation,...,capex_intensity,capex_flag,fcf_conversion,cfo_sign,cfi_sign,cff_sign,capital_pattern,computed_roe,roe_diff,computed_roce
0,0,ABB,Dec 2012,1653,1451,202.0,12.0,33,0,19,...,3.569268,MODERATE,20.792079,+,-,-,REINVESTOR,22.411128,22.411128,31.221020
1,1,ABB,Mar 2014,2276,2009,267.0,12.0,49,0,22,...,6.326889,MODERATE,4.119850,+,-,-,REINVESTOR,25.126904,25.126904,33.883249
2,1,ABB,Mar 2014,2276,2009,267.0,12.0,49,0,22,...,0.000000,ASSET_LIGHT,0.000000,-,-,-,PRE_REVENUE,25.126904,25.126904,33.883249
3,2,ABB,Mar 2015,2289,1977,312.0,14.0,48,0,15,...,8.169506,CAPITAL_INTENSIVE,8.974359,+,-,-,REINVESTOR,24.439701,24.439701,33.297759
4,2,ABB,Mar 2015,2289,1977,312.0,14.0,48,0,15,...,81.432940,CAPITAL_INTENSIVE,-608.653846,-,-,+,DEBT_FUNDED_GROWTH,24.439701,24.439701,33.297759


In [69]:
#final db
pd.read_sql("SELECT COUNT(*) FROM financial_ratios", conn)

,COUNT(*)
0,1210
